### Creating a logistic Regression to predict absenteeism

#### Import the lib

In [ ]:
import pandas as pd
import numpy as np

### Loding the preprocessed data

In [ ]:
data_preprocessed = pd.read_csv('Absenteeism_preprocessed.csv')

In [ ]:
data_preprocessed.head()

#### Create targets

In [ ]:
data_preprocessed['Absenteeism Time in Hours'].median()

**What are these `Absenteeism Time in Hours` for**
- `Moderate absent`: <=3 hrs this indicates -> `0`
- `Excessively absent`: >4 hrs and this indicates -> `1`
- So, **the supervised learning is to predict we get `0's or 1's: Target`**

In [ ]:
# since th median is 3 so, >3 gives 1 or else 0
targets = np.where(data_preprocessed['Absenteeism Time in Hours']> 
                   data_preprocessed['Absenteeism Time in Hours'].median(), 1, 0)

- `np.where(condition, value if True, value if False)`: checks if a condition has been satisfied and assigns a value accordingly

In [ ]:
targets

In [ ]:
data_preprocessed['Excessive Absenteeism'] = targets
data_preprocessed.head()

#### Comment on targets

In [ ]:
# this is targets.sum() has total no of 1's and targets.shape[0] gives the total 
targets.sum()/ targets.shape[0]

- so, aroung 46% of the targets are 1's
- and aroung 54% of the targets are 0's

#### Creating a Checkpoint
but by areating another variable for the dataframe

In [ ]:
# we are droping the value Absenteeism Time in Hours as we already have Excessive absenteeism
data_with_targets = data_preprocessed.drop(['Absenteeism Time in Hours','Daily Work Load Average','Distance to Work','Day of the Week'], axis=1)

In [ ]:
data_with_targets

In [ ]:
data_with_targets is data_preprocessed

it is false because `data_with_targets` have 1 less value than `data_preprocessed`

In [ ]:
data_with_targets.shape

#### Select the inputs for the Regression

`DataFrame.iloc[row, col]:` selects (slice) data by position when given rows and columns wanted pandas

In [ ]:
# since 'Excessive Absenteeism' is the target variable so, it is excluded from input variable
# .iloc: excludes the ending index
data_with_targets.iloc[:,:14]

In [ ]:
# or this
data_with_targets.iloc[:,:-1]

In [ ]:
unscaled_inputs = data_with_targets.iloc[:,:-1]

### Standarize the data

In [ ]:
#from sklearn.preprocessing import StandardScaler

#absenteeism_scaler = StandardScaler() # empty scaler object

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler

class CustomScaler(BaseEstimator, TransformerMixin):

    def __init__(self, columns, copy=True, with_mean=True, with_std=True):
        # Use keyword arguments to pass to StandardScaler
        #but in tutorial its like this
        #self.scaler = StandardScaler(copy,with_mean,with_std) -> which was not working
        self.copy = copy
        self.with_mean = with_mean
        self.with_std = with_std
        self.scaler = StandardScaler(copy=self.copy, with_mean=self.with_mean, with_std=self.with_std)
        self.columns = columns
        self.mean = None
        self.var = None

    def fit(self, X, y=None):
        self.scaler.fit(X[self.columns], y)
        self.mean_ = np.mean(X[self.columns])
        self.var = np.var(X[self.columns])
        return self

    def transform(self, X, y=None, copy=None):
        init_col_order = X.columns
        X_scaled = pd.DataFrame(self.scaler.transform(X[self.columns]), columns=self.columns)
        X_not_scaled = X.loc[:, ~X.columns.isin(self.columns)] #~ inverts the mask, so you get columns not in self.columns.
        return pd.concat([X_not_scaled, X_scaled], axis=1)[init_col_order]


In [ ]:
unscaled_inputs.columns.values

In [ ]:
# obmitting the dummy variables
# columns_to_scale = ['Month Value','Day of the Week', 'Transportation Expense', 'Distance to Work',
      # 'Age', 'Daily Work Load Average', 'Body Mass Index', 'Education','Children', 'Pets']
columns_to_omit = ['Reason_1', 'Reason_2', 'Reason_3', 'Reason_4', 'Education',
       'Children', 'Pets']

**List Comprehension** : is a syntatic construct which allows us to create a list from existing list based on loops, conditionals, etc.

In [ ]:
columns_to_scale = [x for x in unscaled_inputs.columns.values if x not in columns_to_omit]

In [ ]:
absenteeism_scaler = CustomScaler(columns_to_scale)

`absenteeism_scaler`: will be used to substract the mean and divide by the standard deviation variable-wise(feature-wise)

In [ ]:
# this line calculates the mean and sd
absenteeism_scaler.fit(unscaled_inputs)

In [ ]:
# this transform the unscaled input to scaled one
scaled_inputs = absenteeism_scaler.transform(unscaled_inputs)

In [ ]:
scaled_inputs

In [ ]:
scaled_inputs.shape

### Spliting the data into training and testing and shuffle

#### Import the relevent modeule

In [ ]:
from sklearn.model_selection import train_test_split

#### Split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(scaled_inputs, targets, train_size =0.8, random_state= 20)

`train_test_split(scaled_inputs, targets, train_size =0.8, shuffle= True)`: here we don't have to shuffle  
- `random_state`: it shuffle the data in same random way 

In [ ]:
print(x_train.shape, y_train.shape)  

In [ ]:
print(x_test.shape, y_test.shape) 

### Logistic regression with sklearn 

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn import metrics # to evaluate the model

#### Training the model

In [ ]:
reg = LogisticRegression()

In [ ]:
reg.fit(x_train, y_train)

In [ ]:
# accuracy
reg.score(x_train, y_train)

#### Manually checking the accuracy

In [ ]:
model_outputs = reg.predict(x_train)
model_outputs

In [ ]:
y_train

In [ ]:
model_outputs == y_train

In [ ]:
# total no of correct prediction
np.sum(model_outputs == y_train)

In [ ]:
model_outputs.shape[0]

In [ ]:
# Accuarcy = correct prediction/ observation
np.sum(model_outputs == y_train)/ model_outputs.shape[0]

#### Intercept and Coefficients

In [ ]:
reg.intercept_

In [ ]:
reg.coef_

In [ ]:
# scaled_inputs.columns.values

- when unscaled_inputs which are in pandas DataFrame to convert it to scaled it becomes ndarray.

In [ ]:
feature_name = unscaled_inputs.columns.values

In [ ]:
summary_table = pd.DataFrame(columns=['Feature name'], data = feature_name)

summary_table['Coefficient'] = np.transpose(reg.coef_)

summary_table

In [ ]:
summary_table.index = summary_table.index + 1
summary_table.loc[0] = ['Intercept', reg.intercept_[0]]
summary_table = summary_table.sort_index()
summary_table

#### Interpreting the coefficients

In [ ]:
summary_table['Odd_ratios'] = np.exp(summary_table.Coefficient)
summary_table

`DataFrame.sort_values(series)`: sorts the values in a dataframe with resoect to given col (series)

In [ ]:
summary_table.sort_values('Odd_ratios', ascending = False)

- if a coeffiecients is around 0 or odd ratio is around is around 1: **a feature is not particularly important**.
    - it means that a Weights of 0 implies that no matter the feature value, we will multiply it by 0(in the model)
    - For a unit change in the standardization feature, the odds increase by a multiple equal to the odds ratio(1= no change)

- so, in here `Daily Work Load Average`, `Distance to Work`, `Day of the Week` coefficient is nearly 0 & Odd_ratio is aroung 1. so, we would drop this variable as having it doesn't make any diff.

Furthermore, we made a methodological error by standardizing our dummy variables. Dummy variables inherently represent binary categorical states (0 or 1) and lack a continuous scale. While standardization is essential for continuous features to bring them to a common scale, applying it to dummies strips away their intuitive interpretability without adding any mathematical value. Dummies are already naturally scaled.


### Testing the model

In [ ]:
# Accuracy of test dataset
reg.score(x_test, y_test)

if the `test accuracy` is 10-20% < than `train accuracy` -> **is due to overfitting**

`let's check the PROBABLITY of output being 0 or 1 instead of 0 or 1`
**sklearn.linear_model.LogisticRegression.predict_proba(x)**: returns the probablity estimates for all possible outputs

In [ ]:
predicted_proba = reg.predict_proba(x_test)
predicted_proba

In [ ]:
predicted_proba.shape

We, only need the [1] as it says the probablity of 1 and probablity of absenteeism

In [ ]:
predicted_proba[:,1]

#### Saving the model

EASY WAY:
- using **`pickle[module]`**: it is a python module used to convert a python obj into character stream.(basically will save the reg.function which have the logistic regression done and use it to new notebook)

In [ ]:
import pickle

In [ ]:
with open('model', 'wb') as file: #file name: model, write bytes:wb
    pickle.dump(reg, file) # save: dump('specify the obj to be dumped')

In [ ]:
with open('scaler', 'wb') as file: #file name: model, write bytes:wb
    pickle.dump(absenteeism_scaler, file)